In [1]:
from google.colab import drive
drive.mount('/content/drive')



MessageError: Error: credential propagation was unsuccessful

In [ ]:
!mkdir -p /content/drive/MyDrive/brain_tumor_dataset


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/drive/MyDrive/brain_tumor_dataset


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/brain_tumor_dataset/brain-tumor-mri-dataset.zip"
extract_path = "/content/drive/MyDrive/brain_tumor_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted to:", extract_path)


KeyboardInterrupt: 

In [ ]:
import os

base_path = "/content/drive/MyDrive/brain_tumor_dataset"
print(os.listdir(base_path))


['Testing', 'Training', 'sample_train', 'sample_train1']


In [ ]:
import os

train_dir = "/content/drive/MyDrive/brain_tumor_dataset/Training"
test_dir = "/content/drive/MyDrive/brain_tumor_dataset/Testing"


In [ ]:
print("Training folders:")
for folder in os.listdir(train_dir):
    print(folder, "→", len(os.listdir(os.path.join(train_dir, folder))), "images")

print("\nTesting folders:")
for folder in os.listdir(test_dir):
    print(folder, "→", len(os.listdir(os.path.join(test_dir, folder))), "images")


Training folders:
glioma → 1321 images
meningioma → 1339 images
notumor → 1595 images
pituitary → 1457 images

Testing folders:
glioma → 299 images
meningioma → 306 images
notumor → 405 images
pituitary → 300 images


In [ ]:
import os, random, shutil

# Original training data
train_dir = "/content/drive/MyDrive/brain_tumor_dataset/Training"

# New sampled folder
sample_train_dir = "/content/drive/MyDrive/brain_tumor_dataset/sample_train1"
os.makedirs(sample_train_dir, exist_ok=True)

# Loop through each class folder
for cls in os.listdir(train_dir):
    cls_path = os.path.join(train_dir, cls)
    sample_cls_path = os.path.join(sample_train_dir, cls)
    os.makedirs(sample_cls_path, exist_ok=True)

    images = os.listdir(cls_path)
    # Choose 500 random images (or all if less than 500)
    selected_images = random.sample(images, min(500, len(images)))

    for img in selected_images:
        src = os.path.join(cls_path, img)
        dst = os.path.join(sample_cls_path, img)
        shutil.copy(src, dst)

    print(f"✅ Copied {len(selected_images)} images for class: {cls}")


✅ Copied 500 images for class: glioma
✅ Copied 500 images for class: meningioma
✅ Copied 500 images for class: notumor
✅ Copied 500 images for class: pituitary


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
import os



In [ ]:
train_dir = "/content/drive/MyDrive/brain_tumor_dataset/Training"
test_dir = "/content/drive/MyDrive/brain_tumor_dataset/Testing"

In [ ]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 4571 images belonging to 4 classes.
Found 1141 images belonging to 4 classes.


In [ ]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False  # Freeze pretrained layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 111, 111,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 109, 109,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 109, 109,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 54, 54,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 54, 54,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 54,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 54, 54,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 52, 52,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 52,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 52, 52,    │          0 │ batch_normalizat

 Total params: 22,065,572 (84.17 MB)

 Trainable params: 262,788 (1.00 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [ ]:
from PIL import Image
import os

def remove_corrupted_images(directory):
    count = 0
    for root, _, files in os.walk(directory):
        for f in files:
            path = os.path.join(root, f)
            try:
                with Image.open(path) as img:
                    img.verify()  # verify image integrity
            except Exception as e:
                print(f"❌ Removing corrupted file: {path} ({e})")
                os.remove(path)
                count += 1
    print(f"✅ Done. Removed {count} corrupted images.")

remove_corrupted_images('/content/drive/MyDrive/brain_tumor_dataset/sample_train')
remove_corrupted_images('/content/drive/MyDrive/brain_tumor_dataset/Testing')

KeyboardInterrupt: 

In [ ]:
# Recreate the ImageDataGenerators
train_gen = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/brain_tumor_dataset/Training',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    '/content/drive/MyDrive/brain_tumor_dataset/Testing',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 5712 images belonging to 4 classes.


NameError: name 'val_datagen' is not defined

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)

In [ ]:
# Step 1: Load InceptionV3 and freeze initially
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# --- Step 1: Load base model and freeze initially ---
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(4, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

# --- Step 2: Train only top layers (warm-up) ---
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    verbose=1
)

# --- Step 3: Unfreeze deeper layers (fine-tuning) ---
for layer in base_model.layers[-100:]:
    layer.trainable = True

# Re-compile with smaller LR to avoid large updates
model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# --- Step 4: Add training callbacks ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
    ModelCheckpoint('/content/drive/MyDrive/best_inceptionv3_tumor.h5', save_best_only=True, monitor='val_accuracy')
]

# --- Step 5: Fine-tune entire model ---
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

# --- Step 6: Save final trained model ---
model.save('/content/drive/MyDrive/final_inceptionv3_tumor.h5')

Epoch 1/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 2369s 13s/step - accuracy: 0.6347 - loss: 0.8913 - val_accuracy: 0.7607 - val_loss: 0.6409
Epoch 2/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 104s 583ms/step - accuracy: 0.8390 - loss: 0.4584 - val_accuracy: 0.8081 - val_loss: 0.5207
Epoch 3/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 105s 587ms/step - accuracy: 0.8591 - loss: 0.3986 - val_accuracy: 0.8414 - val_loss: 0.4407
Epoch 4/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 105s 585ms/step - accuracy: 0.8708 - loss: 0.3613 - val_accuracy: 0.8510 - val_loss: 0.4137
Epoch 5/5
179/179 ━━━━━━━━━━━━━━━━━━━━ 105s 588ms/step - accuracy: 0.8813 - loss: 0.3390 - val_accuracy: 0.8475 - val_loss: 0.3904
Epoch 1/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 565ms/step - accuracy: 0.7169 - loss: 0.7139

179/179 ━━━━━━━━━━━━━━━━━━━━ 163s 735ms/step - accuracy: 0.7174 - loss: 0.7127 - val_accuracy: 0.8493 - val_loss: 0.4113 - learning_rate: 1.0000e-05
Epoch 2/10
147/179 ━━━━━━━━━━━━━━━━━━━━ 15s 500ms/step - accuracy: 0.8948 - loss: 0.3046